# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [8]:
import pandas as pd
import numpy as np

raw_path = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
initial_rows = len(raw_path)
display(raw_path.head(10))
print(initial_rows)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


30000


In [ ]:
# cleaning the numeric coloumn and filling missing data with zero

numeric_columns = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct"
]

for col in numeric_columns:
    raw_path[col] = pd.to_numeric(raw_path[col], errors='coerce').fillna(0)


# cleaning the categorical column and fill the mission data with unknow
categorical_columns = [
    "content_type", "main_intent", "competition_level", "age_tier", 
    "freshness_tier", "word_count_tier", "char_count_tier", "impression_tier", 
    "position_tier", "trend_direction"
]

for col in categorical_columns:
    raw_path[col] = raw_path[col].fillna('unknown').astype(str).replace({"": "unknow", "nan": "unknow"})

# Applying business filter and removing duplicates

raw_path = raw_path[(raw_path['impressions_90d'] > 0) & (raw_path['content_age_days'] >= 90)].copy()

raw_path = raw_path.drop_duplicates(subset= ['content_id']).reset_index(drop=True)

raw_path['is_declining_label'] = (raw_path['trend_direction'].str.lower() == 'down').astype(int)


# Engineering to handle heavy traffic distribution

raw_path["log_impressions_90d"] = np.log1p(raw_path["impressions_90d"])

raw_path["log_clicks_90d"] = np.log1p(raw_path["clicks_90d"])

raw_path["log_sessions_90d"] = np.log1p(raw_path["sessions_90d"])

raw_path["log_ai_sessions_90d"] = np.log1p(raw_path["ai_sessions_90d"])

# some additional features indicator

raw_path["has_clicks"] = (raw_path["clicks_90d"] > 0).astype(int)

raw_path["has_ai_sessions"] = (raw_path["ai_sessions_90d"] > 0).astype(int)

raw_path["measurable_opportunity"] = ((raw_path["impressions_90d"] >= 100) & (raw_path["sessions_90d"] > 0)).astype(int)

print(f'New data shape {raw_path.shape[0]}')

if raw_path.shape[0] == initial_rows:
    print("No Duplicate rows in the data")
else:
    print(f'Duplicate row detectected in the data are {initial_rows - raw_path.shape[0]}')



New data shape 30000
No Duplicate rows in the data


In [15]:
raw_path.shape
raw_path.to_csv('../../data/cleaned_content_data/cleaned_content_data.csv', index=False)

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

To explain the meaning of each feature we must define its source and verify that it represents data available befors decision points 

**Search console and analytics features**
-- log_impression_90d, ctr, avg_position, scroll_rate
**Meaning** - They measure historical search performance and user interactions
**Missingness** - to be filled with zero
**Available-when** - This represents historical behaviour and is safe to use

**Keyword Context** 
-- search_volumn, competition, cpc, main_intent
**Meaning** - Intent and search market estimates for target terms
**Missingness** - Systematically missing for feedly articles. Should be imputed as 0 or "unknown"
**Available When** - Set during page setup and creation

**Content Propertics**
-- word_count, content_age_days, days_since_last_update
**Meaning** - measures metadetails about the article 
**Missingness** - 0 for word_count
**Available-when** - Readily available in our data

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

clean_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
] 

# clean model
test_feature_clean = raw_path[clean_features]
test_target = raw_path['is_declining_label']

X_train_clean, X_test_clean, y_train, y_test = train_test_split(test_feature_clean, test_target, test_size=0.2, random_state=42)

model_clean = DecisionTreeClassifier(max_depth=5, random_state=42)

model_clean.fit(X_train_clean, y_train)
y_pred_clean = model_clean.predict_proba(X_test_clean)[:, 1]

auc_clean = roc_auc_score(y_test, y_pred_clean)

# leaky model
leaky_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "trend_pct"
] 

test_feature_leaky = raw_path[leaky_features]

X_train_leaky, X_test_leaky, y_train, y_test = train_test_split(test_feature_leaky, test_target, test_size=0.2, random_state=42)

model_leaky = DecisionTreeClassifier(max_depth=5, random_state=42)

model_leaky.fit(X_train_leaky, y_train)

y_pred_leaky = model_leaky.predict_proba(X_test_leaky)[:, 1]

auc_leaky = roc_auc_score(y_test, y_pred_leaky)

print(f"Clean Model AUC: {auc_clean:.4f}")
print(f"Leaky Model AUC: {auc_leaky:.4f}")



Clean Model AUC: 0.7328
Leaky Model AUC: 0.9993


### The leaky model has an ROC-AUC of 0.99 which is nearly 1.00 because the trend_pct tells the model excatly what to predict.
### The clean model has an ROC-AUC od 0.73 

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* **trend_pct** and **trend_direction**: excluded because they are used to define the target label
* **impression_last_30d**, **clicks_last_30d**, and **sessions_last_30d**: Excluded because they cover the recent 30 days of traffic which overlaps with the feature window to predict
* **impression_prev_30d**, **clicks_prev_30d**, and **sessions_prev_30d** : Excluded because the rep basline of the data.
* **provider_used** and **model_used** Excluded because information about the LLM used is not important to the search performance
* **content_id** and **client_id** Excluded because hashed identifies contain no numeric signals. using them as features would cause overfitting

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.